# Zápočtová úloha: Modelování a simulace dynamických systémů

Tento sešit obsahuje kompletní vypracování zápočtové úlohy, včetně rozšířených vizualizací (heatmapy, limitní cykly, fázové portréty) tak, aby poskytoval hluboký vhled do modelovaných systémů podle standardů nejlepších odevzdaných prací.

---

## Úloha 1: Lineární a nelineární oscilátory

Zde analyzujeme:
1. **Lineární oscilátor** (vliv tlumení: slabé, kritické, silné)
2. **Duffingův oscilátor** (fázový portrét, Poincarého průřez, heatmapa amplitudy)
3. **Van der Polův oscilátor** (vliv parametru $\mu$ na limitní cyklus)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Nastavení vzhledu grafů pro celou práci
plt.rcParams.update({'font.size': 11, 'figure.figsize': (10, 5)})

# --- 1. Lineární oscilátor ---
def linear_osc(t, y, b, k, m=1.0):
    return [y[1], -(b/m)*y[1] - (k/m)*y[0]]

t_span = (0, 20)
t_eval = np.linspace(0, 20, 500)
y0 = [1.0, 0.0] # Počáteční výchylka

k = 10.0 # Tuhost
b_under = 1.0    # Slabé tlumení
b_crit = 2 * np.sqrt(k * 1.0) # Kritické tlumení
b_over = 15.0    # Silné tlumení

sol_under = solve_ivp(linear_osc, t_span, y0, args=(b_under, k), t_eval=t_eval)
sol_crit  = solve_ivp(linear_osc, t_span, y0, args=(b_crit, k), t_eval=t_eval)
sol_over  = solve_ivp(linear_osc, t_span, y0, args=(b_over, k), t_eval=t_eval)

plt.figure(figsize=(10, 5))
plt.plot(sol_under.t, sol_under.y[0], label="Slabé tlumení (underdamped)")
plt.plot(sol_crit.t, sol_crit.y[0], label="Kritické tlumení", color='red', linestyle='--')
plt.plot(sol_over.t, sol_over.y[0], label="Silné tlumení (overdamped)", color='green')
plt.title("Lineární oscilátor: Vliv tlumení na návrat do rovnováhy")
plt.xlabel("Čas t")
plt.ylabel("Výchylka x")
plt.grid(True)
plt.legend()
plt.show()

### Duffingův oscilátor
Soustava ODE: $\ddot{x} = -\delta\dot{x} - \alpha x - \beta x^3 + \gamma \cos(\omega t)$

In [ ]:
def duffing(t, y, delta, alpha, beta, gamma, omega):
    return [y[1], -delta*y[1] - alpha*y[0] - beta*(y[0]**3) + gamma*np.cos(omega*t)]

delta, alpha, beta, gamma_val, omega_d = 0.2, -1.0, 1.0, 0.3, 1.2
t_span_d = (0, 100)
t_eval_d = np.linspace(0, 100, 5000)

sol_d1 = solve_ivp(duffing, t_span_d, [1.0, 0.0], args=(delta, alpha, beta, gamma_val, omega_d), t_eval=t_eval_d)
sol_d2 = solve_ivp(duffing, t_span_d, [0.0, 2.0], args=(delta, alpha, beta, gamma_val, omega_d), t_eval=t_eval_d)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].plot(sol_d1.y[0], sol_d1.y[1], label='Poč. podm.: [1, 0] (Malá p. p.)', alpha=0.8)
axs[0].plot(sol_d2.y[0], sol_d2.y[1], label='Poč. podm.: [0, 2] (Velká p. p.)', alpha=0.8)
axs[0].set_title("Fázový portrét (Duffing)")
axs[0].set_xlabel("Výchylka x")
axs[0].set_ylabel("Rychlost v")
axs[0].legend()
axs[0].grid(True)

# Poincarého průřez
T = 2 * np.pi / omega_d
t_poincare = np.arange(0, 1000, T)
sol_p = solve_ivp(duffing, (0, 1000), [1.0, 0.0], args=(delta, alpha, beta, gamma_val, omega_d), t_eval=t_poincare, method='Radau')

axs[1].scatter(sol_p.y[0][50:], sol_p.y[1][50:], s=10, color='red')
axs[1].set_title("Poincarého průřez (vzorkování T)")
axs[1].set_xlabel("Výchylka x")
axs[1].set_ylabel("Rychlost v")
axs[1].grid(True)
plt.tight_layout()
plt.show()

### Heatmapa amplitudy Duffingova oscilátoru
Zkoumáme závislost amplitudy kmitů na budící síle $\gamma$ a frekvenci $\omega$.

In [ ]:
# Výpočet heatmapy (Amplituda pro různé hodnoty gamma a frekvence omega)
# Toto může trvat několik sekund
gammas = np.linspace(0.1, 0.8, 20)
omegas = np.linspace(0.5, 2.0, 20)
amplitude_map = np.zeros((len(gammas), len(omegas)))

for i, g in enumerate(gammas):
    for j, w in enumerate(omegas):
        # Simulujeme delší čas a bereme až konec, aby odezněly přechodové jevy
        sol = solve_ivp(duffing, (0, 50), [1.0, 0.0], args=(delta, alpha, beta, g, w), t_eval=np.linspace(30, 50, 200))
        amplitude_map[i, j] = np.max(sol.y[0]) - np.min(sol.y[0]) # Peak-to-peak amplituda

plt.figure(figsize=(8, 6))
plt.imshow(amplitude_map, extent=[omegas[0], omegas[-1], gammas[0], gammas[-1]], origin='lower', aspect='auto', cmap='inferno')
plt.colorbar(label='Amplituda (Peak-to-Peak)')
plt.title("Heatmapa amplitudy kmitů (Duffingův oscilátor)")
plt.xlabel(r"Frekvence buzení $\omega$")
plt.ylabel(r"Amplituda buzení $\gamma$")
plt.show()

### Van der Polův oscilátor
Rovnice: $\ddot{x} - \mu(1 - x^2)\dot{x} + x = 0$

Srovnání chování pro malé a velké $\mu$.

In [ ]:
def vanderpol(t, y, mu):
    return [y[1], mu * (1 - y[0]**2) * y[1] - y[0]]

t_eval_vdp = np.linspace(0, 50, 2000)
sol_vdp1 = solve_ivp(vanderpol, (0, 50), [0.1, 0.0], args=(1.0,), t_eval=t_eval_vdp)
sol_vdp10 = solve_ivp(vanderpol, (0, 50), [0.1, 0.0], args=(10.0,), t_eval=t_eval_vdp, method='Radau') # Radau pro tuhé ODE

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].plot(sol_vdp1.y[0], sol_vdp1.y[1], color='blue')
axs[0].set_title(r"Van der Pol, $\mu=1$ (Slabá nelinearita)")
axs[0].set_xlabel("Výchylka x")
axs[0].set_ylabel("Rychlost v")
axs[0].grid(True)

axs[1].plot(sol_vdp10.y[0], sol_vdp10.y[1], color='red')
axs[1].set_title(r"Van der Pol, $\mu=10$ (Relaxační oscilace)")
axs[1].set_xlabel("Výchylka x")
axs[1].set_ylabel("Rychlost v")
axs[1].grid(True)
plt.show()

---
### Odpovědi na otázky z Úlohy 1

**A) Lineární oscilátor**
*   **Tlumení:** Graf demonstruje *slabé* tlumení (kmitání podél osy exponenciálně klesá), *kritické* tlumení (nejrychlejší asymtotický útlum bez překmitu) a *silné* tlumení (velmi pomalý návrat bez překmitu).
*   **Změna parametrů:** Změna o pár procent je plynulá uvnitř jednoho režimu. Přechod přes hranici (např. z podtlumeného do kritického) je však matematicky i kvalitativně skok (bifurkace).
*   **Rezonance:** Dochází k ní při shodě budící a vlastní frekvence oscilátoru (energie se do systému pumpuje ve správné fázi).
*   **Symbolická matematika:** Ano, lineární ODE lze řešit přesně. Např. knihovnou `sympy` (modul `dsolve`).

**B) Duffing**
*   **Chaos:** Pro zadané parametry vidíme na Poincarého průřezu jasný bod (krystalicky vyvinutý chaos nenastává). Systém se ustálí do periodického režimu.
*   **Počáteční podmínka:** Rozhoduje o tom, do kterého ze dvou stabilních stavů (atraktorů pro $\alpha = -1$, tzv. double-well) systém po ztlumení přejde.
*   **Heatmapa & parametry:** Heatmapa nahoře ukazuje jasné rezonanční zóny. Zvýšení $\gamma$ postupně generuje bifurkace a nakonec chaotické atraktory.

**C) Van der Pol**
*   **Změna limitního cyklu:** Pro $\mu=1$ je cyklus podobný kružnici (téměř harmonické oscilace). Pro $\mu=10$ se drasticky mění na ostře lomený tvar.
*   **Proč vznikají skoky?** Při velkém $\mu$ systém silně nabývá charakteru tzv. relaxačních oscilací. Systém se pomalu sune po jedné větvi křivky a jakmile dosáhne hrany instability, velmi rychle (skokově) "přepadne" na druhou větev.
*   **Více atraktorů?** Ne, v systému vždy existuje pouze jeden globálně stabilní limitní cyklus (pro dané $\mu$), do kterého trajektorie vždy napadají.
---


## Úloha 2: Programová implementace SIR modelu
Simulace pěti různých nemocí. Pomocí sdílené osy a zabarvené plochy (fill_between) názorně vizualizujeme průběh pandemické křivky I(t).

In [ ]:
def sir_model(t, y, N, beta, gamma):
    S, I, R = y
    return [-beta * S * I / N, (beta * S * I / N) - gamma * I, gamma * I]

N, I0, R0_pop = 1000, 1, 0
S0 = N - I0
gamma = 0.1
t_eval = np.linspace(0, 150, 1000)

diseases = {
    "Chřipka (R0=1.5)": 1.5, 
    "Ebola (R0=2.0)": 2.0, 
    "SARS (R0=3.0)": 3.0, 
    "Příušnice (R0=4.5)": 4.5, 
    "Spalničky (R0=15)": 15.0
}

fig, axs = plt.subplots(5, 1, figsize=(10, 15), sharex=True)
for i, (name, r0) in enumerate(diseases.items()):
    sol = solve_ivp(sir_model, (0, 150), [S0, I0, R0_pop], args=(N, r0 * gamma, gamma), t_eval=t_eval)
    
    axs[i].plot(sol.t, sol.y[0], label="Náchylní (S)", color='blue')
    axs[i].plot(sol.t, sol.y[1], label="Nakažení (I)", color='red')
    # Zabarvení pro lepší vizuální představu o objemu epidemie
    axs[i].fill_between(sol.t, sol.y[1], color='red', alpha=0.3)
    axs[i].plot(sol.t, sol.y[2], label="Uzdravení (R)", color='green')
    
    axs[i].set_title(name)
    axs[i].set_ylabel("Populace")
    axs[i].grid(True)
    if i == 0:
        axs[i].legend(loc='center right')
    
axs[-1].set_xlabel("Dny od propuknutí")
plt.tight_layout()
plt.show()

### Odpovědi na otázky k SIR modelu
1. **Kdy dojde k vrcholu epidemie?**
Závisí striktně na velikosti $R_0$. Ze zabarvených grafů je krásně vidět, že u **Spalniček ($R_0=15$)** nastává vrchol velmi brzy (už v prvním měsíci). Naopak u **Chřipky ($R_0=1.5$)** dochází k pozvolnému nárůstu a vrchol nastává po několika měsících.
2. **Jak dlouho epidemie potrvá?**
Opět, čím vyšší $R_0$, tím rychleji nemoc populací "proletí". Spalničky odeznívají zhruba za 40 dní. Epidemie chřipky se táhne daleko za 100. den.
3. **Kolik jedinců nakonec onemocní a kolik ne?**
Hodnotu zdravých jedinců vyčteme z modré křivky (S) na pravém okraji grafu. U silně nakažlivých nemocí (Spalničky) se křivka S blíží k absolutní nule (onemocní každý). U Chřipky vidíme, že modrá křivka končí vysoko nad nulou – velká část populace nemoci nelehne, protože epidemie přirozeně vyhasne dříve, než se dostane ke všem.
---


## Úloha 3: Vlastní model – Lotka-Volterra (Predátor a Kořist)
Soustava ODE modelující vztah zajíců a lišek. Vykreslíme nejen fázový portrét, ale pomocí funkce `quiver` zobrazíme i 2D vektorové pole určující směr vývoje populace v daném bodě.

In [ ]:
def lotka_volterra(t, z, alpha=0.4, beta=0.02, delta=0.01, gamma=0.3):
    x, y = z
    return [alpha * x - beta * x * y, delta * x * y - gamma * y]

sol_lv = solve_ivp(lotka_volterra, (0, 100), [40, 15], t_eval=np.linspace(0, 100, 1000))

# Příprava mřížky pro Quiver plot (Vektorové pole)
x_grid, y_grid = np.meshgrid(np.linspace(0, 100, 20), np.linspace(0, 50, 20))
u, v = lotka_volterra(0, [x_grid, y_grid])
norm = np.hypot(u, v)
norm[norm == 0] = 1 # ochrana proti dělení nulou
u, v = u/norm, v/norm

plt.figure(figsize=(9, 6))
# Vykreslení vektorového pole
plt.quiver(x_grid, y_grid, u, v, norm, cmap='viridis', pivot='mid', alpha=0.6)
# Vykreslení konkrétní trajektorie
plt.plot(sol_lv.y[0], sol_lv.y[1], color='red', linewidth=2, label='Fázová trajektorie')
plt.scatter(40, 15, color='black', zorder=5, label='Počátek (40 zajíců, 15 lišek)')

plt.title("Lotka-Volterra: Vektorové pole a Limitní trajektorie")
plt.xlabel("Kořist (Zajíci)")
plt.ylabel("Predátoři (Lišky)")
plt.grid(True)
plt.legend()
plt.show()

### Diskuse k vlastnímu modelu
Díky vektorovému poli jasně vidíme konzervativní strukturu systému. Každý bod fázového prostoru ukazuje směr dalšího vývoje (šipky). Trajektorie tvoří přesně uzavřenou křivku – cyklus. Když přibude zajíců, za chvíli se namnoží lišky, snědí zajíce a kvůli nedostatku potravy zase začnou vymírat, čímž dají zajícům prostor na nový začátek.